# EDA y Preprocesamiento - Avance 2
**Proyecto:** Emisiones atmosféricas de fuentes difusas en Chile (2019-2024)

El objetivo de este cuaderno es procesar los datos obtenidos del Registro de Emisiones y Transferencias de Contaminantes (RETC). 

Para responder a la pregunta de nuestro proyecto y garantizar que la aplicación interactiva (Streamlit) funcione de manera fluida, realizaremos las siguientes tareas de calidad y limpieza de datos:
1. Integración de múltiples años y formatos (CSV y Excel).
2. Resolución de inconsistencias de formato (comas decimales y tipos de datos mixtos).
3. Reducción de dimensionalidad (agrupación de más de 4 millones de registros).
4. Filtrado del alcance geográfico (Zona Centro-Sur).

## 1. Configuración del Entorno y Alcance
Importamos las librerías necesarias y definimos las rutas de trabajo respetando la estructura del repositorio (`data/raw/` para los datos originales y `data/processed/` para el resultado limpio). También acotamos nuestro alcance a la zona Centro-Sur del país.

In [1]:
import pandas as pd
import os

# Definimos las rutas
ruta_raw = "../data/raw/"
ruta_processed = "../data/processed/"

# Lista de las regiones de la zona centro-sur
regiones_centro_sur = [
    'Maule', 
    'Ñuble', 
    'Biobío', 
    'Araucanía', 
    'Los Ríos', 
    'Los Lagos'
]

## 2. Carga de Datos y Manejo de Formatos Mixtos
**Decisión de Calidad de Datos:** Al inspeccionar los datos originales, detectamos dos problemas:
* **Formatos inconsistentes:** Los años 2019 a 2024 vienen en `.csv`, pero el año 2023 está en `.xlsx`.
* **Problemas de tipo de variable:** Las columnas geográficas (`Lat`, `Lon`) y la categoría `Rural o Urbano` tienen valores mixtos y muchos nulos. Para evitar advertencias y pérdida de datos en la carga, forzamos la lectura de estas columnas como texto (`str`) desde el inicio.

In [2]:
# Forzamos que las columnas problemáticas se lean como texto (string)
tipos = {'Lat': str, 'Lon': str, 'Rural o Urbano': str}

print("Cargando archivos CSV...")
df_2019 = pd.read_csv(ruta_raw + "ruea-efd-2019-ckan.csv", sep=";", dtype=tipos)
df_2020 = pd.read_csv(ruta_raw + "ruea-efd-2020-ckan.csv", sep=";", dtype=tipos)
df_2021 = pd.read_csv(ruta_raw + "ruea-efd-2021-ckan.csv", sep=";", dtype=tipos)
df_2022 = pd.read_csv(ruta_raw + "ruea-efd-2022-ckan.csv", sep=";", dtype=tipos)
df_2024 = pd.read_csv(ruta_raw + "ruea-efd-2024-ckan.csv", sep=";", dtype=tipos)

print("Cargando archivo Excel de 2023 (esto puede tardar unos segundos más)...")
df_2023 = pd.read_excel(ruta_raw + "efd-ckan-2023.xlsx", dtype=tipos)

# Unimos todos los años en un solo gran DataFrame
df_total = pd.concat([df_2019, df_2020, df_2021, df_2022, df_2023, df_2024], ignore_index=True)

print(f"¡Carga exitosa! Total de registros crudos: {len(df_total)}")

Cargando archivos CSV...
Cargando archivo Excel de 2023 (esto puede tardar unos segundos más)...
¡Carga exitosa! Total de registros crudos: 4198441


## 3. Limpieza Estructural y Transformación Numérica
Para poder realizar cálculos sobre la variable objetivo ($Y$ = Cantidad de Toneladas), es obligatorio transformar el tipo de dato.
**Problema detectado:** El dataset oficial utiliza comas (`,`) como separador decimal, lo que provoca que Pandas lo interprete como texto. 
**Solución:** Estandarizamos los nombres de las columnas para evitar errores de sintaxis, reemplazamos las comas por puntos y transformamos la columna de toneladas a formato matemático (`float64`).

In [3]:
# Estandarizar nombres de columnas: todo a minúsculas y espacios por guiones bajos
df_total.columns = df_total.columns.str.lower().str.strip().str.replace(' ', '_')

# Arreglar los números: reemplazar comas por puntos y convertir a formato float
df_total['cantidad_toneladas'] = df_total['cantidad_toneladas'].astype(str).str.replace(',', '.').astype(float)

# Verificamos que la columna se haya transformado correctamente
print("Tipos de datos actualizados:")
print(df_total[['cantidad_toneladas', 'año', 'region', 'tipo_fuente']].dtypes)

Tipos de datos actualizados:
cantidad_toneladas    float64
año                     int64
region                    str
tipo_fuente               str
dtype: object


## 4. Reducción de Dimensionalidad (Agrupación)
Actualmente contamos con casi 4.2 millones de registros. Cargar este volumen en Streamlit colapsaría la memoria de la aplicación.

**Decisión Analítica:** Agruparemos los datos sumando las toneladas emitidas, basándonos en nuestras variables explicativas y temporales ($X$ y $T$). 
*Nota:* Utilizamos el parámetro `dropna=False` en la agrupación. Esto es vital porque los eventos de "Incendios Forestales" no poseen clasificación de "Rural o Urbano" (tienen valores `NaN`). Si no incluimos este parámetro, Pandas eliminaría todos los registros de incendios de nuestro análisis.

In [4]:
# Definimos las columnas por las que queremos agrupar
columnas_agrupar = [
    'año', 'region', 'provincia', 'comuna', 'id_comuna', 
    'tipo_fuente', 'contaminantes', 'rural_o_urbano'
]

# Agrupamos y sumamos. El dropna=False es vital para no perder las filas de Incendios Forestales
df_resumido = df_total.groupby(columnas_agrupar, dropna=False, as_index=False)['cantidad_toneladas'].sum()

print(f"Reducción completada. Pasamos de {len(df_total)} filas a {len(df_resumido)} filas.")

Reducción completada. Pasamos de 4198441 filas a 104738 filas.


## 5. Aplicación del Alcance y Exportación
Finalmente, filtramos el dataset resultante para quedarnos exclusivamente con las regiones de la zona Centro-Sur (nuestro alcance definido en el Avance 1). Este archivo consolidado para la aplicación interactiva.

In [5]:
# Filtramos solo las regiones que te interesan investigar
df_final = df_resumido[df_resumido['region'].isin(regiones_centro_sur)]

print(f"Registros finales para la zona Centro-Sur: {len(df_final)}")

# Aseguramos que la carpeta processed exista (por si acaso)
os.makedirs(ruta_processed, exist_ok=True)

# Guardamos el archivo final ligero
ruta_guardado = ruta_processed + "emisiones_centro_sur_2019_2024.csv"
df_final.to_csv(ruta_guardado, index=False)

print(f" Archivo listo y guardado en: {ruta_guardado}")
print("Este es el archivo que debes subir a GitHub y conectar a Streamlit.")

Registros finales para la zona Centro-Sur: 53440
 Archivo listo y guardado en: ../data/processed/emisiones_centro_sur_2019_2024.csv
Este es el archivo que debes subir a GitHub y conectar a Streamlit.
